In [ ]:
code = """
#include <stdio.h>  // Standard C library for input and output functions (like printf)

// ----------------------------------------------------------
// CUDA Kernel Function
// ----------------------------------------------------------
// The '__global__' keyword tells the compiler that this function
// will run on the GPU (device) but can be called from the CPU (host).
__global__ void add(int *a, int *b, int *c, int n) {
    // Calculate the unique thread ID
    // 'threadIdx.x' = index of the thread inside its block
    // 'blockIdx.x' = index of the block in the grid
    // 'blockDim.x' = number of threads per block
    int tid = threadIdx.x + blockIdx.x * blockDim.x;

    // Each thread handles one element of the array
    // The 'if' condition ensures we don’t access out-of-bound elements
    if (tid < n) {
        c[tid] = a[tid] + b[tid];  // Perform addition for the thread’s corresponding index
    }
}

// ----------------------------------------------------------
// Main Function - Host Code (runs on CPU)
// ----------------------------------------------------------
int main() {
    const int N = 10;  // Total number of elements in each array

    // Declare host (CPU) arrays
    int a[N], b[N], c[N];

    // Declare device (GPU) pointers
    int *d_a, *d_b, *d_c;

    // Initialize arrays 'a' and 'b' with values
    for (int i = 0; i < N; i++) {
        a[i] = i;       // Array A: 0, 1, 2, 3, ...
        b[i] = i * i;   // Array B: 0, 1, 4, 9, ...
    }

    // ----------------------------------------------------------
    // Allocate memory on the GPU (device)
    // ----------------------------------------------------------
    // cudaMalloc() allocates memory on the GPU and returns a pointer to it.
    cudaMalloc(&d_a, N * sizeof(int));  // Allocate memory for array a
    cudaMalloc(&d_b, N * sizeof(int));  // Allocate memory for array b
    cudaMalloc(&d_c, N * sizeof(int));  // Allocate memory for result array c

    // ----------------------------------------------------------
    // Copy data from CPU (host) to GPU (device)
    // ----------------------------------------------------------
    // cudaMemcpy(destination, source, size_in_bytes, direction)
    cudaMemcpy(d_a, a, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, N * sizeof(int), cudaMemcpyHostToDevice);

    // ----------------------------------------------------------
    // Launch the CUDA kernel on the GPU
    // ----------------------------------------------------------
    // Syntax: kernel_name<<<number_of_blocks, number_of_threads_per_block>>>(parameters);
    // Here, we launch 1 block containing N threads.
    add<<<1, N>>>(d_a, d_b, d_c, N);

    // ----------------------------------------------------------
    // Copy the result back from GPU to CPU
    // ----------------------------------------------------------
    cudaMemcpy(c, d_c, N * sizeof(int), cudaMemcpyDeviceToHost);

    // ----------------------------------------------------------
    // Display the result on the CPU
    // ----------------------------------------------------------
    printf("Result:\n");
    for (int i = 0; i < N; i++) {
        printf("%d + %d = %d\n", a[i], b[i], c[i]);
    }

    // ----------------------------------------------------------
    // Free the allocated GPU memory
    // ----------------------------------------------------------
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;  // End of program
}

"""

# Save it to a file
with open("add.cu", "w") as f:
    f.write(code)

In [ ]:
!nvcc -arch=sm_75 add.cu -o add

In [ ]:
!./add

Result:
0 + 0 = 0
1 + 1 = 2
2 + 4 = 6
3 + 9 = 12
4 + 16 = 20
5 + 25 = 30
6 + 36 = 42
7 + 49 = 56
8 + 64 = 72
9 + 81 = 90


In [ ]:
!cp add.cu /content/drive/MyDrive/GPU_Programming/

The `add.cu` file has been copied to your Google Drive in the root directory. You can change the destination path in the code above if you want to save it in a different folder.